# Longitudinal Severity Analysis — B2AI Voice Dataset
Load sessions, diagnoses, and questionnaire severity scores, then explore and plot.

In [ ]:
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

# ── Paths ──────────────────────────────────────────────────────────
DATA_ROOT = Path(os.path.expanduser(
    "~/B2AI/physionet.org/files/b2ai-voice/3.0.0"
))
DIAG_DIR  = DATA_ROOT / "phenotype" / "diagnosis"
TASK_DIR  = DATA_ROOT / "phenotype" / "task"
QUEST_DIR = DATA_ROOT / "phenotype" / "questionnaire"
FEAT_DIR  = DATA_ROOT / "features"
mel_spec_path = FEAT_DIR / "torchaudio_mel_spectrogram.parquet"


def load_tsv(p):
    return pd.read_csv(p, sep="\t", dtype=str)

print(f"Data root: {DATA_ROOT}")
print(f"Exists:    {DATA_ROOT.exists()}")

Data root: /home/srl/B2AI/physionet.org/files/b2ai-voice/3.0.0
Exists:    True


## 0. Comprehensive Data Loading
Load **all** tables: features (static + MFCC), phenotype demographics, every
diagnosis file, enrollment tables, all questionnaires, and all task tables.

In [196]:
# ══════════════════════════════════════════════════════════════════
#  A.  FEATURES
# ══════════════════════════════════════════════════════════════════

# A1. Static acoustic features (TSV)
static_features_df = pd.read_csv(FEAT_DIR / "static_features.tsv", sep="\t")
static_features_df["participant_id"] = static_features_df["participant_id"].astype(str)
static_features_df["session_id"]     = static_features_df["session_id"].astype(str)
print(f"[features/static_features]  {static_features_df.shape}")

# A2. MFCC features (Parquet — 60-coeff MFCCs per recording)
mfcc_df = pd.read_parquet(FEAT_DIR / "torchaudio_mfcc.parquet")
mfcc_df["participant_id"] = mfcc_df["participant_id"].astype(str)
mfcc_df["session_id"]     = mfcc_df["session_id"].astype(str)
print(f"[features/torchaudio_mfcc]  {mfcc_df.shape}")

# ══════════════════════════════════════════════════════════════════
#  B.  PHENOTYPE — Demographics
# ══════════════════════════════════════════════════════════════════
DEMO_DIR = DATA_ROOT / "phenotype" / "demographics"

demographics_df = load_tsv(DEMO_DIR / "demographics.tsv")
print(f"\n[phenotype/demographics]    {demographics_df.shape}")

# ══════════════════════════════════════════════════════════════════
#  C.  PHENOTYPE — Diagnosis  (one TSV per condition)
# ══════════════════════════════════════════════════════════════════
ENROLL_DIR = DATA_ROOT / "phenotype" / "enrollment"

diagnosis_dfs = {}
for f in sorted(DIAG_DIR.glob("*.tsv")):
    name = f.stem                         # e.g. "parkinsons_disease"
    diagnosis_dfs[name] = load_tsv(f)
    print(f"[diagnosis/{name}]  {diagnosis_dfs[name].shape}")

# Also build a combined long-form diagnosis table
_diag_rows = []
for name, df in diagnosis_dfs.items():
    tmp = df[["participant_id"]].copy()
    tmp["diagnosis"] = name
    _diag_rows.append(tmp)
all_diagnoses_df = pd.concat(_diag_rows, ignore_index=True)
all_diagnoses_df["participant_id"] = all_diagnoses_df["participant_id"].astype(str)
print(f"\n[diagnosis/combined]        {all_diagnoses_df.shape}  "
      f"({all_diagnoses_df['diagnosis'].nunique()} conditions)")

# ══════════════════════════════════════════════════════════════════
#  D.  PHENOTYPE — Enrollment
# ══════════════════════════════════════════════════════════════════
participant_df     = load_tsv(ENROLL_DIR / "participant.tsv")
enrollment_form_df = load_tsv(ENROLL_DIR / "enrollment_form.tsv")
eligibility_df     = load_tsv(ENROLL_DIR / "eligibility.tsv")
print(f"\n[enrollment/participant]     {participant_df.shape}")
print(f"[enrollment/enrollment_form] {enrollment_form_df.shape}")
print(f"[enrollment/eligibility]     {eligibility_df.shape}")

# ══════════════════════════════════════════════════════════════════
#  E.  PHENOTYPE — Questionnaires  (one TSV per instrument)
# ══════════════════════════════════════════════════════════════════
questionnaire_dfs = {}
for f in sorted(QUEST_DIR.glob("*.tsv")):
    name = f.stem
    questionnaire_dfs[name] = load_tsv(f)
    print(f"[questionnaire/{name}]  {questionnaire_dfs[name].shape}")

# ══════════════════════════════════════════════════════════════════
#  F.  PHENOTYPE — Task  (session, recording, acoustic_task, etc.)
# ══════════════════════════════════════════════════════════════════
task_dfs = {}
for f in sorted(TASK_DIR.glob("*.tsv")):
    name = f.stem
    task_dfs[name] = load_tsv(f)
    print(f"[task/{name}]  {task_dfs[name].shape}")

print("\n✅  All tables loaded.")

[features/static_features]  (32249, 135)
[features/torchaudio_mfcc]  (29020, 5)

[phenotype/demographics]    (911, 61)
[diagnosis/adhd_adult]  (173, 21)
[diagnosis/airway_stenosis]  (138, 32)
[diagnosis/amyotrophic_lateral_sclerosis]  (3, 36)
[diagnosis/anxiety]  (18, 21)
[diagnosis/benign_lesions]  (58, 23)
[diagnosis/bipolar_disorder]  (12, 25)
[diagnosis/cognitive_impairment]  (74, 33)
[diagnosis/control]  (148, 2)
[diagnosis/copd_and_asthma]  (9, 72)
[diagnosis/depression]  (44, 21)
[diagnosis/glottic_insufficiency]  (8, 27)
[diagnosis/laryngeal_cancer]  (4, 44)
[diagnosis/laryngeal_dystonia]  (78, 41)
[diagnosis/laryngitis]  (6, 40)
[diagnosis/muscle_tension_dysphonia]  (58, 35)
[diagnosis/parkinsons_disease]  (106, 58)
[diagnosis/precancerous_lesions]  (14, 44)
[diagnosis/psychiatric_history]  (84, 5)
[diagnosis/ptsd_adult]  (141, 12)
[diagnosis/unexplained_chronic_cough]  (41, 32)
[diagnosis/unilateral_vocal_fold_paralysis]  (69, 33)

[diagnosis/combined]        (1286, 2)  (21 c

In [197]:
# ── Quick summary of all loaded DataFrames ─────────────────────────
summary_rows = []

# Features
summary_rows.append(("features/static_features", *static_features_df.shape))
summary_rows.append(("features/torchaudio_mfcc", *mfcc_df.shape))

# Demographics
summary_rows.append(("phenotype/demographics", *demographics_df.shape))

# Diagnosis (individual + combined)
for name, df in diagnosis_dfs.items():
    summary_rows.append((f"diagnosis/{name}", *df.shape))
summary_rows.append(("diagnosis/COMBINED", *all_diagnoses_df.shape))

# Enrollment
summary_rows.append(("enrollment/participant", *participant_df.shape))
summary_rows.append(("enrollment/enrollment_form", *enrollment_form_df.shape))
summary_rows.append(("enrollment/eligibility", *eligibility_df.shape))

# Questionnaires
for name, df in questionnaire_dfs.items():
    summary_rows.append((f"questionnaire/{name}", *df.shape))

# Tasks
for name, df in task_dfs.items():
    summary_rows.append((f"task/{name}", *df.shape))

summary = pd.DataFrame(summary_rows, columns=["table", "rows", "columns"])
print(f"{'─'*60}")
print(f"  LOADED {len(summary)} tables  |  "
      f"Total rows across all: {summary['rows'].sum():,}")
print(f"{'─'*60}")
display(summary.style.hide(axis="index"))

────────────────────────────────────────────────────────────
  LOADED 47 tables  |  Total rows across all: 121,055
────────────────────────────────────────────────────────────


table,rows,columns
features/static_features,32249,135
features/torchaudio_mfcc,29020,5
phenotype/demographics,911,61
diagnosis/adhd_adult,173,21
diagnosis/airway_stenosis,138,32
diagnosis/amyotrophic_lateral_sclerosis,3,36
diagnosis/anxiety,18,21
diagnosis/benign_lesions,58,23
diagnosis/bipolar_disorder,12,25
diagnosis/cognitive_impairment,74,33


## 1. Filter to Longitudinal Participants (≥ 2 sessions)
Use the **session** table to count unique sessions per participant, keep only
those with **session_count ≥ 2**, then filter every loaded table accordingly.

In [198]:
# ── Count sessions per participant from the session table ──────────
session_df = task_dfs["session"].copy()
session_df["participant_id"] = session_df["participant_id"].astype(str)
session_df["session_id"]     = session_df["session_id"].astype(str)

sess_counts = (session_df
               .groupby("participant_id")["session_id"]
               .nunique()
               .rename("n_sessions"))

# Keep only participants with >= 2 sessions
longitudinal_pids = set(sess_counts[sess_counts >= 2].index)

print(f"Total participants:               {len(sess_counts)}")
print(f"Longitudinal (≥ 2 sessions):      {len(longitudinal_pids)}")
print(f"\nSession-count distribution (longitudinal only):")
print(sess_counts[sess_counts >= 2]
      .value_counts().sort_index()
      .to_frame("n_participants"))

Total participants:               833
Longitudinal (≥ 2 sessions):      143

Session-count distribution (longitudinal only):
            n_participants
n_sessions                
2                       98
3                       24
4                       12
5                        6
6                        2
7                        1


In [ ]:
# ── Helper: filter any DataFrame that has a participant_id column ──
def filter_longitudinal(df, pids=longitudinal_pids):
    """Return rows whose participant_id is in the longitudinal set."""
    pid_col = df["participant_id"].astype(str)
    return df[pid_col.isin(pids)].copy()

# ── Apply filter to every loaded table ─────────────────────────────

# Features
static_features_df = filter_longitudinal(static_features_df)
mfcc_df            = filter_longitudinal(mfcc_df)

# Demographics
demographics_df = filter_longitudinal(demographics_df)

# Diagnosis (individual files + combined)
for name in diagnosis_dfs:
    diagnosis_dfs[name] = filter_longitudinal(diagnosis_dfs[name])
all_diagnoses_df = filter_longitudinal(all_diagnoses_df)

# Enrollment
participant_df     = filter_longitudinal(participant_df)
enrollment_form_df = filter_longitudinal(enrollment_form_df)
eligibility_df     = filter_longitudinal(eligibility_df)

# Questionnaires
for name in questionnaire_dfs:
    questionnaire_dfs[name] = filter_longitudinal(questionnaire_dfs[name])

# Tasks
for name in task_dfs:
    task_dfs[name] = filter_longitudinal(task_dfs[name])

# Session table itself
session_df = filter_longitudinal(session_df)

# ── Print summary after filtering ──────────────────────────────────
print(f"After filtering to {len(longitudinal_pids)} longitudinal participants:\n")

filtered_summary = [
    ("features/static_features", *static_features_df.shape),
    ("features/torchaudio_mfcc", *mfcc_df.shape),
    ("phenotype/demographics",   *demographics_df.shape),
    ("diagnosis/COMBINED",       *all_diagnoses_df.shape),
    ("enrollment/participant",   *participant_df.shape),
    ("enrollment/enrollment_form", *enrollment_form_df.shape),
    ("enrollment/eligibility",   *eligibility_df.shape),
]
for name, df in questionnaire_dfs.items():
    filtered_summary.append((f"questionnaire/{name}", *df.shape))
for name, df in task_dfs.items():
    filtered_summary.append((f"task/{name}", *df.shape))

fs = pd.DataFrame(filtered_summary, columns=["table", "rows", "columns"])

After filtering to 143 longitudinal participants:

────────────────────────────────────────────────────────────
  26 tables  |  Total rows: 29,748
────────────────────────────────────────────────────────────


table,rows,columns
features/static_features,7451,135
features/torchaudio_mfcc,7477,5
phenotype/demographics,200,61
diagnosis/COMBINED,284,2
enrollment/participant,143,2
enrollment/enrollment_form,143,28
enrollment/eligibility,143,42
questionnaire/custom_affect_scale,59,16
questionnaire/dsm5_adult,43,41
questionnaire/dyspnea_index,54,13


In [200]:
# ── Add visit number to session table ──────────────────────────────
session_df["visit_num"] = (session_df
                           .groupby("participant_id")
                           .cumcount() + 1)

## 1.5 Raw Severity Score Distributions (Before Any Filtering)
For every diagnosis table, show the **raw value counts** and **NaN counts** of each
severity / score / degree column — so we can see exactly what we have before
section 2 converts and filters.

In [201]:
# ══════════════════════════════════════════════════════════════════
#  RAW severity scores → TXT file, grouped by participant_id
#  (BEFORE any conversion or filtering)
# ══════════════════════════════════════════════════════════════════

SCORE_COLUMNS = {
    "airway_stenosis": [
        "diagnosis_as_ds_ac", "diagnosis_as_ds_mcg", "diagnosis_as_ds_ods",
        "diagnosis_as_ds_eps", "diagnosis_as_ds_bc",
    ],
    "benign_lesions": [
        "diagnosis_bl_degree_os_1", "diagnosis_bl_degree_r_1",
        "diagnosis_bl_degree_b_1",  "diagnosis_bl_degree_s_1",
        "diagnosis_bl_degree_p_1",  "diagnosis_bl_degree_l_1",
        "diagnosis_bl_degree_os_2", "diagnosis_bl_degree_r_2",
        "diagnosis_bl_degree_b_2",  "diagnosis_bl_degree_s_2",
        "diagnosis_bl_degree_p_2",  "diagnosis_bl_degree_l_2",
    ],
    "cognitive_impairment": [
        "diagnosis_alz_dementia_mci_ds_cdr",
    ],
    "copd_and_asthma": [
        "diagnosis_ca_cat_score", "diagnosis_ca_mmrc_grade_score",
        "diagnosis_ca_act_score", "diagnosis_ca_gold_group",
    ],
    "depression": [
        "diagnosis_bipolar_severity_depression",
    ],
    "glottic_insufficiency": [
        "diagnosis_gi_degree_os_1", "diagnosis_gi_degree_r_1",
        "diagnosis_gi_degree_b_1",  "diagnosis_gi_degree_s_1",
        "diagnosis_gi_degree_p_1",  "diagnosis_gi_degree_l_1",
        "diagnosis_gi_degree_os_2", "diagnosis_gi_degree_r_2",
        "diagnosis_gi_degree_b_2",  "diagnosis_gi_degree_s_2",
        "diagnosis_gi_degree_p_2",  "diagnosis_gi_degree_l_2",
    ],
    "laryngeal_cancer": [
        "diagnosis_lc_ds_t_stage", "diagnosis_lc_ds_n_stage",
        "diagnosis_lc_ds_m_stage",
        "diagnosis_lc_degree_os_1", "diagnosis_lc_degree_r_1",
        "diagnosis_lc_degree_b_1",  "diagnosis_lc_degree_s_1",
        "diagnosis_lc_degree_p_1",  "diagnosis_lc_degree_l_1",
        "diagnosis_lc_degree_os_2", "diagnosis_lc_degree_r_2",
        "diagnosis_lc_degree_b_2",  "diagnosis_lc_degree_s_2",
        "diagnosis_lc_degree_p_2",  "diagnosis_lc_degree_l_2",
    ],
    "laryngeal_dystonia": [
        "diagnosis_ld_degree_os_1", "diagnosis_ld_degree_r_1",
        "diagnosis_ld_degree_b_1",  "diagnosis_ld_degree_s_1",
        "diagnosis_ld_degree_p_1",  "diagnosis_ld_degree_l_1",
        "diagnosis_ld_degree_os_2", "diagnosis_ld_degree_r_2",
        "diagnosis_ld_degree_b_2",  "diagnosis_ld_degree_s_2",
        "diagnosis_ld_degree_p_2",  "diagnosis_ld_degree_l_2",
    ],
    "laryngitis": [
        "diagnosis_l_degree_os_1", "diagnosis_l_degree_r_1",
        "diagnosis_l_degree_b_1",  "diagnosis_l_degree_s_1",
        "diagnosis_l_degree_p_1",  "diagnosis_l_degree_l_1",
        "diagnosis_l_degree_os_2", "diagnosis_l_degree_r_2",
        "diagnosis_l_degree_b_2",  "diagnosis_l_degree_s_2",
        "diagnosis_l_degree_p_2",  "diagnosis_l_degree_l_2",
    ],
    "muscle_tension_dysphonia": [
        "diagnosis_mtd_degree_os_1", "diagnosis_mtd_degree_r_1",
        "diagnosis_mtd_degree_b_1",  "diagnosis_mtd_degree_s_1",
        "diagnosis_mtd_degree_p_1",  "diagnosis_mtd_degree_l_1",
        "diagnosis_mtd_degree_os_2", "diagnosis_mtd_degree_r_2",
        "diagnosis_mtd_degree_b_2",  "diagnosis_mtd_degree_s_2",
        "diagnosis_mtd_degree_p_2",  "diagnosis_mtd_degree_l_2",
    ],
    "parkinsons_disease": [
        "diagnosis_parkinsons_ds",
    ],
    "precancerous_lesions": [
        "diagnosis_pl_degree_os_1", "diagnosis_pl_degree_r_1",
        "diagnosis_pl_degree_b_1",  "diagnosis_pl_degree_s_1",
        "diagnosis_pl_degree_p_1",  "diagnosis_pl_degree_l_1",
        "diagnosis_pl_degree_os_2", "diagnosis_pl_degree_r_2",
        "diagnosis_pl_degree_b_2",  "diagnosis_pl_degree_s_2",
        "diagnosis_pl_degree_p_2",  "diagnosis_pl_degree_l_2",
    ],
    "unilateral_vocal_fold_paralysis": [
        "diagnosis_degree_os",  "diagnosis_degree_r",
        "diagnosis_degree_b",   "diagnosis_degree_s",
        "diagnosis_degree_p",   "diagnosis_degree_l",
        "diagnosis_degree_os_2", "diagnosis_degree_r_2",
        "diagnosis_degree_b_2",  "diagnosis_degree_s_2",
        "diagnosis_degree_p_2",  "diagnosis_degree_l_2",
    ],
}

OUT_FILE = Path("/home/srl/B2AI/LLM/raw_severity_scores.txt")
lines = []
W = lambda s: lines.append(s)

# ════════════════════════════════════════════════════════════════
#  PART 1 — Value counts per illness × score column
# ════════════════════════════════════════════════════════════════
W("=" * 80)
W("  PART 1: VALUE COUNTS PER ILLNESS × SCORE COLUMN  (raw, before filtering)")
W("=" * 80)

for name, cols in SCORE_COLUMNS.items():
    if name not in diagnosis_dfs:
        W(f"\n⚠  {name} — not in diagnosis_dfs (0 longitudinal rows), skipping")
        continue

    df_raw = diagnosis_dfs[name]
    present_cols = [c for c in cols if c in df_raw.columns]
    if not present_cols:
        W(f"\n⚠  {name} — none of the score columns found, skipping")
        continue

    total_rows = len(df_raw)
    n_pids = df_raw["participant_id"].nunique()

    W(f"\n{'═' * 75}")
    W(f"  {name.upper().replace('_', ' ')}  "
      f"({total_rows} rows, {n_pids} participants, {len(present_cols)} score cols)")
    W(f"{'═' * 75}")

    for col in present_cols:
        n_na = df_raw[col].isna().sum()
        n_valid = total_rows - n_na
        pct_na = 100 * n_na / total_rows if total_rows > 0 else 0

        W(f"\n  ── {col}")
        W(f"     NaN: {n_na}/{total_rows} ({pct_na:.0f}%)  |  Valid: {n_valid}")

        vc = df_raw[col].dropna().value_counts()
        if len(vc) == 0:
            W(f"     (all values are NaN)")
        elif len(vc) <= 20:
            for val, cnt in vc.items():
                W(f"     {cnt:>5d}  {val}")
        else:
            numeric_vals = pd.to_numeric(df_raw[col], errors="coerce").dropna()
            if len(numeric_vals) > 0:
                W(f"     Numeric range: {numeric_vals.min():.1f} – "
                  f"{numeric_vals.max():.1f}  "
                  f"(mean={numeric_vals.mean():.1f}, "
                  f"median={numeric_vals.median():.1f})")
                W(f"     {len(vc)} unique values")
            else:
                for val, cnt in vc.head(15).items():
                    W(f"     {cnt:>5d}  {val}")
                if len(vc) > 15:
                    W(f"     ... and {len(vc) - 15} more unique values")

# ════════════════════════════════════════════════════════════════
#  PART 2 — Diagnosis scores grouped by participant_id
#  NOTE: Diagnosis tables have NO session_id (1 row per participant)
#        → cannot track the same diagnosis metric over time
# ════════════════════════════════════════════════════════════════
W(f"\n\n{'=' * 80}")
W("  PART 2: DIAGNOSIS SEVERITY SCORES GROUPED BY PARTICIPANT_ID")
W("  ⚠ Diagnosis tables have NO session_id — 1 row per participant (not longitudinal)")
W("=" * 80)

pid_records = {}
for name, cols in SCORE_COLUMNS.items():
    if name not in diagnosis_dfs:
        continue
    df_raw = diagnosis_dfs[name]
    present_cols = [c for c in cols if c in df_raw.columns]
    if not present_cols:
        continue
    for _, row in df_raw.iterrows():
        pid = str(row["participant_id"])
        for col in present_cols:
            val = row[col]
            val_str = str(val) if pd.notna(val) else "NaN"
            if pid not in pid_records:
                pid_records[pid] = []
            pid_records[pid].append((name, col, val_str))

# Show only participants with > 1 valid score value
longitudinal_candidates = {}
for pid in sorted(pid_records.keys()):
    records = pid_records[pid]
    diseases = {}
    for disease, col, val in records:
        diseases.setdefault(disease, []).append((col, val))
    n_valid = sum(1 for _, c, v in records if v != "NaN")
    if n_valid > 1:
        longitudinal_candidates[pid] = {
            "n_valid": n_valid, "n_diseases": len(diseases),
            "diseases": diseases, "n_total": len(records),
        }

W(f"\n  Participants with > 1 valid diagnosis score: "
  f"{len(longitudinal_candidates)} / {len(pid_records)}\n")
for pid in sorted(longitudinal_candidates.keys()):
    info = longitudinal_candidates[pid]
    W(f"\n{'─' * 75}")
    W(f"  participant_id: {pid}")
    W(f"  Diagnoses: {info['n_diseases']}  |  "
      f"Score values: {info['n_valid']} valid / {info['n_total']} total")
    W(f"{'─' * 75}")
    for disease, entries in info["diseases"].items():
        valid_entries = [(c, v) for c, v in entries if v != "NaN"]
        if valid_entries:
            W(f"\n    [{disease}]  ({len(valid_entries)} valid scores)")
            for col, val in valid_entries:
                W(f"        {col:55s} = {val}")

# ════════════════════════════════════════════════════════════════
#  PART 3 — LONGITUDINAL: Questionnaire scores with session_id
#  These tables DO have session_id → same metric across sessions
# ════════════════════════════════════════════════════════════════
W(f"\n\n{'=' * 80}")
W("  PART 3: LONGITUDINAL QUESTIONNAIRE SCORES (same metric, multiple sessions)")
W("  ✔ These tables HAVE session_id → true longitudinal data")
W("=" * 80)

# Map questionnaire → (session_id column, score columns to track)
QUEST_SCORES = {
    "vhi10": {
        "session_col": "vhi_session_id",
        "score_cols": ["vhi_10_calc_score"],
        "description": "Voice Handicap Index (VHI-10 total score)",
    },
    "phq9": {
        "session_col": "phq_9_session_id",
        "score_cols": [
            "no_interest", "feeling_depressed", "trouble_sleeping",
            "no_energy", "no_appetite", "feeling_bad_self",
            "trouble_concentrate", "move_speak_slow", "thoughts_death",
        ],
        "description": "PHQ-9 Depression (9 items, each 0–3)",
    },
    "gad7_anxiety": {
        "session_col": "gad_7_session_id",
        "score_cols": [
            "nervous_anxious", "cant_control_worry", "worry_too_much",
            "trouble_relaxing", "hard_to_sit_still", "easily_agitated",
            "afraid_of_things",
        ],
        "description": "GAD-7 Anxiety (7 items, each 0–3)",
    },
    "voice_perception": {
        "session_col": "voice_perception_session_id",
        "score_cols": ["voice_quality_perception"],
        "description": "Self-rated voice quality perception",
    },
    "dyspnea_index": {
        "session_col": "dyspnea_index_session_id",
        "score_cols": [
            "di_effort_breathe", "di_air_in", "di_strain",
            "di_tightness_throat", "di_sound_breathing_in",
            "di_breathing_worse_stress", "di_breathing_stresses_me",
            "di_exercise", "di_restrict_personal_social_life",
            "di_weather_changes",
        ],
        "description": "Dyspnea Index (10 items, breathing difficulty)",
    },
    "leicester_cough_questionnaire": {
        "session_col": "leicester_cough_session_id",
        "score_cols": [
            "lcq_bout", "lcq_sputum_phlegm", "lcq_energy",
            "lcq_chest_stomach_pains", "lcq_sleep", "lcq_tired",
            "lcq_felt_in_control", "lcq_embarrassed",
            "lcq_frustrated", "lcq_fed_up", "lcq_hoarse_voice",
            "lcq_anxious", "lcq_interfere_job",
            "lcq_interfere_life", "lcq_exposure_paint",
            "lcq_interrupt_conversation", "lcq_other_people",
            "lcq_serious_illness", "lcq_partner",
        ],
        "description": "Leicester Cough Questionnaire (19 items)",
    },
}

# Collect participants with ≥ 2 sessions per questionnaire
quest_longitudinal = {}  # quest_name → {pid: [(session_id, {col: val}), ...]}

for qname, qcfg in QUEST_SCORES.items():
    if qname not in questionnaire_dfs:
        continue
    df_q = questionnaire_dfs[qname]
    sess_col = qcfg["session_col"]
    score_cols = [c for c in qcfg["score_cols"] if c in df_q.columns]
    if not score_cols or sess_col not in df_q.columns:
        continue

    # Group by participant, count sessions
    grouped = df_q.groupby("participant_id")
    multi_session = {}
    for pid, grp in grouped:
        n_sess = grp[sess_col].nunique()
        if n_sess >= 2:
            sessions = []
            for _, row in grp.iterrows():
                sid = str(row[sess_col])
                vals = {c: str(row[c]) if pd.notna(row[c]) else "NaN" for c in score_cols}
                sessions.append((sid, vals))
            multi_session[str(pid)] = sessions

    if multi_session:
        quest_longitudinal[qname] = multi_session

    W(f"\n{'═' * 75}")
    W(f"  {qname.upper().replace('_', ' ')}  —  {qcfg['description']}")
    W(f"  Participants with ≥ 2 sessions: {len(multi_session)}")
    W(f"  Score columns: {score_cols}")
    W(f"{'═' * 75}")

    # Value counts for each score column (across all sessions)
    for col in score_cols:
        n_na = df_q[col].isna().sum()
        n_valid = len(df_q) - n_na
        pct_na = 100 * n_na / len(df_q) if len(df_q) > 0 else 0
        W(f"\n  ── {col}")
        W(f"     NaN: {n_na}/{len(df_q)} ({pct_na:.0f}%)  |  Valid: {n_valid}")
        vc = df_q[col].dropna().value_counts()
        if len(vc) <= 20:
            for val, cnt in vc.items():
                W(f"     {cnt:>5d}  {val}")
        else:
            numeric_vals = pd.to_numeric(df_q[col], errors="coerce").dropna()
            if len(numeric_vals) > 0:
                W(f"     Range: {numeric_vals.min():.1f} – {numeric_vals.max():.1f}  "
                  f"(mean={numeric_vals.mean():.1f}, median={numeric_vals.median():.1f})")
            for val, cnt in vc.head(10).items():
                W(f"     {cnt:>5d}  {val}")
            if len(vc) > 10:
                W(f"     ... and {len(vc) - 10} more unique values")

    # Print each longitudinal participant
    for pid in sorted(multi_session.keys()):
        sessions = multi_session[pid]
        W(f"\n    participant_id: {pid}  ({len(sessions)} sessions)")
        for sid, vals in sessions:
            vals_str = "  |  ".join(f"{c}={v}" for c, v in vals.items())
            W(f"      session: {sid[:20]:20s}  {vals_str}")

# ════════════════════════════════════════════════════════════════
#  SUMMARY
# ════════════════════════════════════════════════════════════════
W(f"\n\n{'=' * 80}")
W(f"  SUMMARY: LONGITUDINAL CANDIDATES (same metric, ≥ 2 sessions)")
W(f"{'=' * 80}")

W(f"\n  A) Diagnosis tables: NO session_id → NOT longitudinal (1 row/participant)")
W(f"     Participants with > 1 valid diagnosis score: {len(longitudinal_candidates)}")

W(f"\n  B) Questionnaire tables: HAVE session_id → TRUE longitudinal data")
all_quest_pids = set()
for qname, pid_data in quest_longitudinal.items():
    n = len(pid_data)
    all_quest_pids |= set(pid_data.keys())
    W(f"     {qname:45s}  {n:>3d} participants with ≥ 2 sessions")
W(f"\n     Total unique participants with ≥ 2 sessions (any questionnaire): "
  f"{len(all_quest_pids)}")

W(f"\n  {'participant_id':40s}  questionnaires_with_multi_session")
W(f"  {'─' * 70}")
for pid in sorted(all_quest_pids):
    quests = [qn for qn, pd_ in quest_longitudinal.items() if pid in pd_]
    W(f"  {pid:40s}  {', '.join(quests)}")
W(f"{'=' * 80}")

OUT_FILE.write_text("\n".join(lines), encoding="utf-8")
print(f"✅  Written to {OUT_FILE.resolve()}")
print(f"   {len(lines)} lines, {len(pid_records)} participants")
print(f"   {sum(len(v) for v in pid_records.values())} total score entries across "
      f"{len(SCORE_COLUMNS)} diseases")

✅  Written to /home/srl/B2AI/LLM/raw_severity_scores.txt
   2116 lines, 123 participants
   545 total score entries across 13 diseases


## 2. Keep Only Valid Numeric Severity Scores (Diagnosis Only)
Map **every** diagnosis table to **all** its severity / score columns
(CAPE-V voice quality, TNM staging, Hoehn & Yahr, CDR, Cotton-Myer, etc.),
coerce to numeric, and drop rows where the score is null or non-numeric.

In [202]:
# ══════════════════════════════════════════════════════════════════
#  Map EVERY diagnosis table to ALL its severity / score columns
# ══════════════════════════════════════════════════════════════════
#
# Logic per value in a score column:
#   1. If the value is already a number  →  keep it as-is
#   2. If the value is a string          →  apply the ordinal_map to convert
#   3. If null / unmapped string         →  NaN  →  row gets dropped
#
# CAPE-V dimensions (voice disorders):
#   B = Breathiness, L = Loudness, OS = Overall Severity,
#   P = Pitch, R = Roughness, S = Strain
#   _1 suffix = numeric 0-100 scale, _2 suffix = Consistent/Intermittent

# Shared ordinal map for consistency columns (all voice disorders)
_CONSISTENCY = {"Consistent": 1, "Intermittent": 2, "None": 0}

SEVERITY_MAP = {
    # ─── 1. Airway Stenosis ────────────────────────────────────────
    "airway_stenosis": {
        "score_cols": [
            "diagnosis_as_ds_ac",           # Cotton-Myer grade (Grade 1-4)
            "diagnosis_as_ds_mcg",          # Modified Cotton-Myer grade (Grade I-III)
            "diagnosis_as_ds_ods",          # Overall degree of severity
            "diagnosis_as_ds_eps",          # Endoscopic percent stenosis (numeric)
            "diagnosis_as_ds_bc",           # Bogdasarian classification
            "diagnosis_as_ds_agw",          # Agwatsa classification
        ],
        "ordinal_map": {
            "diagnosis_as_ds_ac": {
                "Grade 1": 1, "Grade 2": 2, "Grade 3": 3, "Grade 4": 4,
            },
            "diagnosis_as_ds_mcg": {
                "Grade I": 1, "Grade II": 2, "Grade III": 3, "Grade IV": 4,
            },
            "diagnosis_as_ds_ods": {
                "Mild": 1, "Moderate": 2, "Severe": 3,
            },
            "diagnosis_as_ds_bc": {
                "(a) Class 1": 1, "(b) Class 2": 2,
                "(c) Class 3": 3, "(d) Class 4": 4,
            },
            "diagnosis_as_ds_agw": {
                "Grade 1 (0-35% obstruction)": 1, "Grade 2 (36-70% obstruction)": 2,
                "Grade 3 (71-100% obstruction)": 3, "Grade 4 (100% obstruction)": 4,
            },
        },
    },

    # ─── 2. Benign Lesions (CAPE-V) ───────────────────────────────
    "benign_lesions": {
        "score_cols": [
            "diagnosis_bl_degree_os_1", "diagnosis_bl_degree_r_1",
            "diagnosis_bl_degree_b_1",  "diagnosis_bl_degree_s_1",
            "diagnosis_bl_degree_p_1",  "diagnosis_bl_degree_l_1",
            "diagnosis_bl_degree_os_2", "diagnosis_bl_degree_r_2",
            "diagnosis_bl_degree_b_2",  "diagnosis_bl_degree_s_2",
            "diagnosis_bl_degree_p_2",  "diagnosis_bl_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_bl_degree_os_2": _CONSISTENCY,
            "diagnosis_bl_degree_r_2":  _CONSISTENCY,
            "diagnosis_bl_degree_b_2":  _CONSISTENCY,
            "diagnosis_bl_degree_s_2":  _CONSISTENCY,
            "diagnosis_bl_degree_p_2":  _CONSISTENCY,
            "diagnosis_bl_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 3. Cognitive Impairment / Alzheimer's / MCI ──────────────
    "cognitive_impairment": {
        "score_cols": ["diagnosis_alz_dementia_mci_ds_cdr"],
        "ordinal_map": {
            "diagnosis_alz_dementia_mci_ds_cdr": {
                "CDR 0 (No impairment)": 0,
                "CDR 0.5 (Questionable)": 0.5,
                "CDR 1 (Mild impairment)": 1,
                "CDR 2 (Moderate impairment)": 2,
                "CDR 3 (Severe impairment)": 3,
            }
        },
    },

    # ─── 4. COPD & Asthma ─────────────────────────────────────────
    "copd_and_asthma": {
        "score_cols": [
            "diagnosis_ca_cat_score",        # CAT score (numeric)
            "diagnosis_ca_mmrc_grade_score",  # mMRC grade (numeric)
            "diagnosis_ca_act_score",         # Asthma Control Test
            "diagnosis_ca_gold_group",        # GOLD group (A/B/C/D)
        ],
        "ordinal_map": {
            "diagnosis_ca_act_score": {
                "Well controlled (≥ 20)": 3,
                "Not controlled (15-19)": 2,
                "Poorly controlled (< 15)": 1,
            },
            "diagnosis_ca_gold_group": {
                "A": 1, "B": 2, "C": 3, "D": 4,
            },
        },
    },

    # ─── 5. Depression ─────────────────────────────────────────────
    "depression": {
        "score_cols": ["diagnosis_bipolar_severity_depression"],
        "ordinal_map": {
            "diagnosis_bipolar_severity_depression": {
                "Mild": 1, "Moderate": 2, "Severe": 3,
            }
        },
    },

    # ─── 6. Glottic Insufficiency (CAPE-V) ────────────────────────
    "glottic_insufficiency": {
        "score_cols": [
            "diagnosis_gi_degree_os_1", "diagnosis_gi_degree_r_1",
            "diagnosis_gi_degree_b_1",  "diagnosis_gi_degree_s_1",
            "diagnosis_gi_degree_p_1",  "diagnosis_gi_degree_l_1",
            "diagnosis_gi_degree_os_2", "diagnosis_gi_degree_r_2",
            "diagnosis_gi_degree_b_2",  "diagnosis_gi_degree_s_2",
            "diagnosis_gi_degree_p_2",  "diagnosis_gi_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_gi_degree_os_2": _CONSISTENCY,
            "diagnosis_gi_degree_r_2":  _CONSISTENCY,
            "diagnosis_gi_degree_b_2":  _CONSISTENCY,
            "diagnosis_gi_degree_s_2":  _CONSISTENCY,
            "diagnosis_gi_degree_p_2":  _CONSISTENCY,
            "diagnosis_gi_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 7. Laryngeal Cancer (TNM staging + CAPE-V) ───────────────
    "laryngeal_cancer": {
        "score_cols": [
            # TNM staging
            "diagnosis_lc_ds_t_stage", "diagnosis_lc_ds_n_stage",
            "diagnosis_lc_ds_m_stage",
            # CAPE-V
            "diagnosis_lc_degree_os_1", "diagnosis_lc_degree_r_1",
            "diagnosis_lc_degree_b_1",  "diagnosis_lc_degree_s_1",
            "diagnosis_lc_degree_p_1",  "diagnosis_lc_degree_l_1",
            "diagnosis_lc_degree_os_2", "diagnosis_lc_degree_r_2",
            "diagnosis_lc_degree_b_2",  "diagnosis_lc_degree_s_2",
            "diagnosis_lc_degree_p_2",  "diagnosis_lc_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_lc_ds_t_stage": {
                "Tis (carcinoma in situ)": 0, "T1a": 1, "T1b": 1,
                "T2": 2, "T3": 3, "T4a": 4, "T4b": 4,
            },
            "diagnosis_lc_ds_n_stage": {
                "N0": 0, "N1": 1, "N2": 2, "N3": 3,
            },
            "diagnosis_lc_ds_m_stage": {
                "M0: no distant metastasis": 0, "M1": 1,
            },
            "diagnosis_lc_degree_os_2": _CONSISTENCY,
            "diagnosis_lc_degree_r_2":  _CONSISTENCY,
            "diagnosis_lc_degree_b_2":  _CONSISTENCY,
            "diagnosis_lc_degree_s_2":  _CONSISTENCY,
            "diagnosis_lc_degree_p_2":  _CONSISTENCY,
            "diagnosis_lc_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 8. Laryngeal Dystonia / Spasmodic Dysphonia (CAPE-V) ─────
    "laryngeal_dystonia": {
        "score_cols": [
            "diagnosis_ld_degree_os_1", "diagnosis_ld_degree_r_1",
            "diagnosis_ld_degree_b_1",  "diagnosis_ld_degree_s_1",
            "diagnosis_ld_degree_p_1",  "diagnosis_ld_degree_l_1",
            "diagnosis_ld_degree_os_2", "diagnosis_ld_degree_r_2",
            "diagnosis_ld_degree_b_2",  "diagnosis_ld_degree_s_2",
            "diagnosis_ld_degree_p_2",  "diagnosis_ld_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_ld_degree_os_2": _CONSISTENCY,
            "diagnosis_ld_degree_r_2":  _CONSISTENCY,
            "diagnosis_ld_degree_b_2":  _CONSISTENCY,
            "diagnosis_ld_degree_s_2":  _CONSISTENCY,
            "diagnosis_ld_degree_p_2":  _CONSISTENCY,
            "diagnosis_ld_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 9. Laryngitis (CAPE-V) ───────────────────────────────────
    "laryngitis": {
        "score_cols": [
            "diagnosis_l_degree_os_1", "diagnosis_l_degree_r_1",
            "diagnosis_l_degree_b_1",  "diagnosis_l_degree_s_1",
            "diagnosis_l_degree_p_1",  "diagnosis_l_degree_l_1",
            "diagnosis_l_degree_os_2", "diagnosis_l_degree_r_2",
            "diagnosis_l_degree_b_2",  "diagnosis_l_degree_s_2",
            "diagnosis_l_degree_p_2",  "diagnosis_l_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_l_degree_os_2": _CONSISTENCY,
            "diagnosis_l_degree_r_2":  _CONSISTENCY,
            "diagnosis_l_degree_b_2":  _CONSISTENCY,
            "diagnosis_l_degree_s_2":  _CONSISTENCY,
            "diagnosis_l_degree_p_2":  _CONSISTENCY,
            "diagnosis_l_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 10. Muscle Tension Dysphonia (CAPE-V) ────────────────────
    "muscle_tension_dysphonia": {
        "score_cols": [
            "diagnosis_mtd_degree_os_1", "diagnosis_mtd_degree_r_1",
            "diagnosis_mtd_degree_b_1",  "diagnosis_mtd_degree_s_1",
            "diagnosis_mtd_degree_p_1",  "diagnosis_mtd_degree_l_1",
            "diagnosis_mtd_degree_os_2", "diagnosis_mtd_degree_r_2",
            "diagnosis_mtd_degree_b_2",  "diagnosis_mtd_degree_s_2",
            "diagnosis_mtd_degree_p_2",  "diagnosis_mtd_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_mtd_degree_os_2": _CONSISTENCY,
            "diagnosis_mtd_degree_r_2":  _CONSISTENCY,
            "diagnosis_mtd_degree_b_2":  _CONSISTENCY,
            "diagnosis_mtd_degree_s_2":  _CONSISTENCY,
            "diagnosis_mtd_degree_p_2":  _CONSISTENCY,
            "diagnosis_mtd_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 11. Parkinson's Disease (Hoehn & Yahr staging) ───────────
    "parkinsons_disease": {
        "score_cols": ["diagnosis_parkinsons_ds"],
        "ordinal_map": {
            "diagnosis_parkinsons_ds": {
                "Stage 1 (Unilateral involvement only)": 1,
                "Stage 2 (Bilateral involvement without impairment of balance)": 2,
                "Stage 3 (Bilateral involvement with mild to moderate impairment of balance)": 3,
                "Stage 4 (Severe disability but still able to walk or stand unassisted)": 4,
                "Stage 5 (Wheelchair-bound or bedridden unless aided)": 5,
            }
        },
    },

    # ─── 12. Precancerous Lesions (CAPE-V) ────────────────────────
    "precancerous_lesions": {
        "score_cols": [
            "diagnosis_pl_degree_os_1", "diagnosis_pl_degree_r_1",
            "diagnosis_pl_degree_b_1",  "diagnosis_pl_degree_s_1",
            "diagnosis_pl_degree_p_1",  "diagnosis_pl_degree_l_1",
            "diagnosis_pl_degree_os_2", "diagnosis_pl_degree_r_2",
            "diagnosis_pl_degree_b_2",  "diagnosis_pl_degree_s_2",
            "diagnosis_pl_degree_p_2",  "diagnosis_pl_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_pl_degree_os_2": _CONSISTENCY,
            "diagnosis_pl_degree_r_2":  _CONSISTENCY,
            "diagnosis_pl_degree_b_2":  _CONSISTENCY,
            "diagnosis_pl_degree_s_2":  _CONSISTENCY,
            "diagnosis_pl_degree_p_2":  _CONSISTENCY,
            "diagnosis_pl_degree_l_2":  _CONSISTENCY,
        },
    },

    # ─── 13. Unilateral Vocal Fold Paralysis (CAPE-V) ─────────────
    #     Note: UVFP uses _b, _l, _os ... (no _1 suffix for numeric)
    "unilateral_vocal_fold_paralysis": {
        "score_cols": [
            "diagnosis_degree_os",  "diagnosis_degree_r",
            "diagnosis_degree_b",   "diagnosis_degree_s",
            "diagnosis_degree_p",   "diagnosis_degree_l",
            "diagnosis_degree_os_2", "diagnosis_degree_r_2",
            "diagnosis_degree_b_2",  "diagnosis_degree_s_2",
            "diagnosis_degree_p_2",  "diagnosis_degree_l_2",
        ],
        "ordinal_map": {
            "diagnosis_degree_os_2": _CONSISTENCY,
            "diagnosis_degree_r_2":  _CONSISTENCY,
            "diagnosis_degree_b_2":  _CONSISTENCY,
            "diagnosis_degree_s_2":  _CONSISTENCY,
            "diagnosis_degree_p_2":  _CONSISTENCY,
            "diagnosis_degree_l_2":  _CONSISTENCY,
        },
    },
}

# Diseases WITHOUT severity scores (kept for reference):
#   adhd_adult, amyotrophic_lateral_sclerosis, anxiety, bipolar_disorder,
#   control, psychiatric_history, ptsd_adult, unexplained_chronic_cough

print(f"SEVERITY_MAP covers {len(SEVERITY_MAP)} diagnosis tables\n")

# ══════════════════════════════════════════════════════════════════
#  Convert each score column: number → keep, string → ordinal map
# ══════════════════════════════════════════════════════════════════
severity_clean = {}

for name, cfg in SEVERITY_MAP.items():
    if name not in diagnosis_dfs:
        print(f"⚠  {name} — table not found (maybe 0 longitudinal rows), skipping")
        continue

    df = diagnosis_dfs[name].copy()
    score_cols = [c for c in cfg["score_cols"] if c in df.columns]
    if not score_cols:
        print(f"⚠  {name} — score columns not in table, skipping")
        continue

    ordinal = cfg.get("ordinal_map", {})

    for col in score_cols:
        raw = df[col]
        # Try to parse every value as a number
        numeric = pd.to_numeric(raw, errors="coerce")
        # For values that failed (strings), apply ordinal map if available
        if col in ordinal:
            still_nan = numeric.isna() & raw.notna()  # was not empty, just non-numeric
            mapped = raw.map(ordinal[col])
            numeric = numeric.where(~still_nan, mapped)
        df[col] = numeric

    # Keep only rows where at least one score column has a valid number
    before = len(df)
    mask = df[score_cols].notna().any(axis=1)
    df_clean = df[mask].copy()
    after = len(df_clean)

    severity_clean[name] = df_clean

    n_score_cols = len(score_cols)
    print(f"✔  {name:35s}  {before:>4d} → {after:>4d} rows  "
          f"(dropped {before - after:>3d})  [{n_score_cols} score cols]")

print(f"\n── {len(severity_clean)} diagnosis tables with valid severity scores ──")

SEVERITY_MAP covers 13 diagnosis tables

✔  airway_stenosis                        12 →   12 rows  (dropped   0)  [6 score cols]
✔  benign_lesions                          9 →    9 rows  (dropped   0)  [12 score cols]
✔  cognitive_impairment                   38 →    1 rows  (dropped  37)  [1 score cols]
✔  copd_and_asthma                         0 →    0 rows  (dropped   0)  [4 score cols]
✔  depression                             10 →    1 rows  (dropped   9)  [1 score cols]
✔  glottic_insufficiency                   1 →    1 rows  (dropped   0)  [12 score cols]
✔  laryngeal_cancer                        0 →    0 rows  (dropped   0)  [14 score cols]
✔  laryngeal_dystonia                      6 →    6 rows  (dropped   0)  [12 score cols]
✔  laryngitis                              0 →    0 rows  (dropped   0)  [12 score cols]
✔  muscle_tension_dysphonia                5 →    4 rows  (dropped   1)  [12 score cols]
✔  parkinsons_disease                     41 →   30 rows  (dropped  11)  

In [205]:
# ══════════════════════════════════════════════════════════════════
#  3. JOIN: Diagnosis severity (static) × Questionnaire scores (longitudinal)
# ══════════════════════════════════════════════════════════════════
#
#  • Diagnosis tables have NO session_id (1 row per participant)
#    → they provide the BASELINE disease severity label
#  • Questionnaire tables HAVE session_id (multiple rows per participant)
#    → they provide REPEATED measures over time
#
#  Strategy:
#    1. Build a single "diagnosis severity" table (1 row per participant)
#    2. Build questionnaire longitudinal tables (≥ 2 sessions per participant)
#    3. Inner join on participant_id → keep only participants who have BOTH
# ══════════════════════════════════════════════════════════════════

# ── Step 1: Unified diagnosis severity table ─────────────────────
#    One row per participant, with disease name + prefixed score columns
diag_rows = []
for disease_name, df in severity_clean.items():
    if len(df) == 0:
        continue
    cfg = SEVERITY_MAP[disease_name]
    score_cols = [c for c in cfg["score_cols"] if c in df.columns]
    for _, row in df.iterrows():
        entry = {
            "participant_id": str(row["participant_id"]),
            "disease": disease_name,
        }
        for col in score_cols:
            entry[f"{disease_name}__{col}"] = row[col]
        diag_rows.append(entry)

diag_severity_df = pd.DataFrame(diag_rows)
diag_pids = set(diag_severity_df["participant_id"].unique())
print(f"Diagnosis severity: {len(diag_severity_df)} rows, "
      f"{len(diag_pids)} unique participants")
print(f"  Diseases present: "
      f"{diag_severity_df['disease'].value_counts().to_dict()}\n")

# ── Step 2: Questionnaire longitudinal tables (≥ 2 sessions) ────
QUEST_SCORES = {
    "vhi10": {
        "session_col": "vhi_session_id",
        "score_cols": ["vhi_10_calc_score"],
    },
    "phq9": {
        "session_col": "phq_9_session_id",
        "score_cols": [
            "no_interest", "feeling_depressed", "trouble_sleeping",
            "no_energy", "no_appetite", "feeling_bad_self",
            "trouble_concentrate", "move_speak_slow", "thoughts_death",
        ],
    },
    "gad7_anxiety": {
        "session_col": "gad_7_session_id",
        "score_cols": [
            "nervous_anxious", "cant_control_worry", "worry_too_much",
            "trouble_relaxing", "hard_to_sit_still", "easily_agitated",
            "afraid_of_things",
        ],
    },
    "voice_perception": {
        "session_col": "voice_perception_session_id",
        "score_cols": ["voice_quality_perception"],
    },
    "dyspnea_index": {
        "session_col": "dyspnea_index_session_id",
        "score_cols": [
            "di_effort_breathe", "di_air_in", "di_strain",
            "di_tightness_throat", "di_sound_breathing_in",
            "di_breathing_worse_stress", "di_breathing_stresses_me",
            "di_exercise", "di_restrict_personal_social_life",
            "di_weather_changes",
        ],
    },
    "leicester_cough_questionnaire": {
        "session_col": "leicester_cough_session_id",
        "score_cols": [
            "lcq_bout", "lcq_sputum_phlegm", "lcq_energy",
            "lcq_chest_stomach_pains", "lcq_sleep", "lcq_tired",
            "lcq_felt_in_control", "lcq_embarrassed",
            "lcq_frustrated", "lcq_fed_up", "lcq_hoarse_voice",
            "lcq_anxious", "lcq_interfere_job",
            "lcq_interfere_life", "lcq_exposure_paint",
            "lcq_interrupt_conversation", "lcq_other_people",
            "lcq_serious_illness", "lcq_partner",
        ],
    },
}

quest_long_dfs = {}  # qname → DataFrame with participant_id, session_id, score_cols

for qname, qcfg in QUEST_SCORES.items():
    if qname not in questionnaire_dfs:
        continue
    df_q = questionnaire_dfs[qname].copy()
    sess_col = qcfg["session_col"]
    score_cols = [c for c in qcfg["score_cols"] if c in df_q.columns]

    if not score_cols or sess_col not in df_q.columns:
        continue

    # Coerce score columns to numeric
    for col in score_cols:
        df_q[col] = pd.to_numeric(df_q[col], errors="coerce")

    # Rename session column to a standard name
    df_q = df_q.rename(columns={sess_col: "session_id"})
    df_q["participant_id"] = df_q["participant_id"].astype(str)
    df_q["session_id"] = df_q["session_id"].astype(str)

    # Keep only participants with ≥ 2 sessions
    sess_per_pid = df_q.groupby("participant_id")["session_id"].nunique()
    pids_multi = set(sess_per_pid[sess_per_pid >= 2].index)
    df_q = df_q[df_q["participant_id"].isin(pids_multi)].copy()

    if len(df_q) == 0:
        continue

    # Prefix score columns with questionnaire name
    rename_map = {c: f"{qname}__{c}" for c in score_cols}
    df_q = df_q.rename(columns=rename_map)

    keep_cols = ["participant_id", "session_id"] + list(rename_map.values())
    quest_long_dfs[qname] = df_q[keep_cols]

    n_pids = df_q["participant_id"].nunique()
    n_rows = len(df_q)
    print(f"✔  {qname:40s}  {n_pids:>3d} participants, "
          f"{n_rows:>4d} rows (≥ 2 sessions), {len(score_cols)} score cols")

# ── Step 3: Merge all questionnaires into one wide longitudinal table ─
if quest_long_dfs:
    # Outer-join all questionnaires on (participant_id, session_id)
    quest_all = None
    for qname, df_q in quest_long_dfs.items():
        if quest_all is None:
            quest_all = df_q
        else:
            quest_all = quest_all.merge(
                df_q, on=["participant_id", "session_id"], how="outer"
            )

    quest_pids = set(quest_all["participant_id"].unique())
    print(f"\nAll questionnaires merged: {quest_all.shape}  "
          f"({len(quest_pids)} unique participants)")
else:
    quest_all = pd.DataFrame(columns=["participant_id", "session_id"])
    quest_pids = set()
    print("\n⚠  No questionnaire longitudinal data found")

# ── Step 4: Inner join — participants with BOTH diagnosis severity ──
#    AND questionnaire longitudinal data
#    → One table PER disease
overlap_pids = diag_pids & quest_pids
print(f"{'═' * 70}")
print(f"  Diagnosis severity participants:     {len(diag_pids)}")
print(f"  Questionnaire longitudinal (≥ 2 s):  {len(quest_pids)}")
print(f"  OVERLAP (both):                      {len(overlap_pids)}")
print(f"{'═' * 70}")

# Build one merged table per disease
disease_tables = {}  # disease_name → DataFrame

for disease_name in diag_severity_df["disease"].unique():
    # Get diagnosis rows for this disease
    diag_sub = diag_severity_df[
        (diag_severity_df["disease"] == disease_name)
        & (diag_severity_df["participant_id"].isin(overlap_pids))
    ].copy()

    if len(diag_sub) == 0:
        continue

    # Keep only the score columns relevant to this disease (drop NaN-only cols)
    diag_score_cols = [c for c in diag_sub.columns
                       if c.startswith(f"{disease_name}__")]
    keep_diag_cols = ["participant_id"] + diag_score_cols
    diag_sub = diag_sub[keep_diag_cols]

    # Merge with questionnaire longitudinal data
    quest_sub = quest_all[quest_all["participant_id"].isin(
        set(diag_sub["participant_id"])
    )].copy()

    if len(quest_sub) == 0:
        continue

    merged = quest_sub.merge(diag_sub, on="participant_id", how="left")

    # Drop questionnaire columns that are entirely NaN for this disease subset
    merged = merged.dropna(axis=1, how="all")

    disease_tables[disease_name] = merged

# ── Print summary per disease table ──────────────────────────────
print(f"\n✅  Created {len(disease_tables)} disease-specific tables:\n")
for disease_name, df in sorted(disease_tables.items()):
    n_pids = df["participant_id"].nunique()
    n_rows = len(df)
    n_cols = len(df.columns)
    n_sessions = df["session_id"].nunique()

    # Split columns into diagnosis vs questionnaire
    diag_cols = [c for c in df.columns if c.startswith(f"{disease_name}__")]
    quest_cols = [c for c in df.columns
                  if "__" in c and not c.startswith(f"{disease_name}__")]

    print(f"{'─' * 70}")
    print(f"  disease_tables['{disease_name}']")
    print(f"  Shape: {df.shape}  |  "
          f"{n_pids} participants × {n_sessions} sessions")
    print(f"  Diagnosis severity cols ({len(diag_cols)}): {diag_cols}")
    print(f"  Questionnaire score cols ({len(quest_cols)}): "
          f"{quest_cols[:5]}{'...' if len(quest_cols) > 5 else ''}")
print(f"{'─' * 70}")

# ── Display each table ───────────────────────────────────────────
for disease_name, df in sorted(disease_tables.items()):
    print(f"\n{'═' * 70}")
    print(f"  {disease_name.upper().replace('_', ' ')}  "
          f"({df['participant_id'].nunique()} participants, {len(df)} rows)")
    print(f"{'═' * 70}")
    display(df)

parkinsons_disease = disease_tables["parkinsons_disease"]

Diagnosis severity: 76 rows, 74 unique participants
  Diseases present: {'parkinsons_disease': 30, 'airway_stenosis': 12, 'unilateral_vocal_fold_paralysis': 10, 'benign_lesions': 9, 'laryngeal_dystonia': 6, 'muscle_tension_dysphonia': 4, 'precancerous_lesions': 2, 'cognitive_impairment': 1, 'depression': 1, 'glottic_insufficiency': 1}

✔  vhi10                                      31 participants,   98 rows (≥ 2 sessions), 1 score cols
✔  phq9                                       31 participants,   99 rows (≥ 2 sessions), 9 score cols
✔  gad7_anxiety                               30 participants,   96 rows (≥ 2 sessions), 7 score cols
✔  voice_perception                           32 participants,  101 rows (≥ 2 sessions), 1 score cols
✔  dyspnea_index                               5 participants,   13 rows (≥ 2 sessions), 10 score cols
✔  leicester_cough_questionnaire               4 participants,   10 rows (≥ 2 sessions), 19 score cols

All questionnaires merged: (103, 49)  (32 uniqu

,participant_id,session_id,voice_perception__voice_quality_perception,laryngeal_dystonia__diagnosis_ld_degree_os_1,laryngeal_dystonia__diagnosis_ld_degree_r_1,laryngeal_dystonia__diagnosis_ld_degree_b_1,laryngeal_dystonia__diagnosis_ld_degree_s_1,laryngeal_dystonia__diagnosis_ld_degree_p_1,laryngeal_dystonia__diagnosis_ld_degree_l_1,laryngeal_dystonia__diagnosis_ld_degree_os_2,laryngeal_dystonia__diagnosis_ld_degree_r_2,laryngeal_dystonia__diagnosis_ld_degree_b_2,laryngeal_dystonia__diagnosis_ld_degree_s_2,laryngeal_dystonia__diagnosis_ld_degree_p_2,laryngeal_dystonia__diagnosis_ld_degree_l_2
0,164394,347143d2,7.0,20.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
1,164394,c5c9d525,3.0,20.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0



══════════════════════════════════════════════════════════════════════
  PARKINSONS DISEASE  (6 participants, 20 rows)
══════════════════════════════════════════════════════════════════════


,participant_id,session_id,vhi10__vhi_10_calc_score,voice_perception__voice_quality_perception,parkinsons_disease__diagnosis_parkinsons_ds
0,225994,45b76d73,10.0,7.0,1.0
1,225994,7ea3f87d,4.0,7.0,1.0
2,225994,b9d3ca96,1.0,7.0,1.0
3,225994,cf3dce17,3.0,7.0,1.0
4,423447,22e21723,29.0,4.0,2.0
5,423447,6d49a73c,29.0,8.0,2.0
6,423447,89d50f25,29.0,4.0,2.0
7,442297,08e3ee65,18.0,7.0,1.0
8,442297,921feddf,21.0,4.0,1.0
9,442297,d6c1b6b3,21.0,8.0,1.0



══════════════════════════════════════════════════════════════════════
  UNILATERAL VOCAL FOLD PARALYSIS  (1 participants, 3 rows)
══════════════════════════════════════════════════════════════════════


,participant_id,session_id,vhi10__vhi_10_calc_score,voice_perception__voice_quality_perception,unilateral_vocal_fold_paralysis__diagnosis_degree_os,unilateral_vocal_fold_paralysis__diagnosis_degree_r,unilateral_vocal_fold_paralysis__diagnosis_degree_b,unilateral_vocal_fold_paralysis__diagnosis_degree_s,unilateral_vocal_fold_paralysis__diagnosis_degree_p,unilateral_vocal_fold_paralysis__diagnosis_degree_l,unilateral_vocal_fold_paralysis__diagnosis_degree_os_2,unilateral_vocal_fold_paralysis__diagnosis_degree_r_2,unilateral_vocal_fold_paralysis__diagnosis_degree_b_2,unilateral_vocal_fold_paralysis__diagnosis_degree_s_2,unilateral_vocal_fold_paralysis__diagnosis_degree_p_2,unilateral_vocal_fold_paralysis__diagnosis_degree_l_2
0,658685,9f023ace,0.0,8.0,34.0,21.0,34.0,0.0,34.0,28.0,1.0,1.0,1.0,1.0,1.0,1.0
1,658685,c68ad108,NaN,NaN,34.0,21.0,34.0,0.0,34.0,28.0,1.0,1.0,1.0,1.0,1.0,1.0
2,658685,cdb3cedc,24.0,4.0,34.0,21.0,34.0,0.0,34.0,28.0,1.0,1.0,1.0,1.0,1.0,1.0


## 4. Join Disease Tables with MFCC & Mel Spectrogram Features
For each disease table, join with the **MFCC** and **Mel Spectrogram** features on
`(participant_id, session_id)`. Each session has multiple recordings (tasks),
so each participant × session row will expand to one row per recording.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  4. Join disease tables with MFCC & Mel Spectrogram features
# ══════════════════════════════════════════════════════════════════

# ── Collect all participant_ids & session_ids we need ─────────────
needed_pids = set()
needed_pairs = set()  # (participant_id, session_id)
for disease_name, df in disease_tables.items():
    for _, row in df.iterrows():
        needed_pids.add(str(row["participant_id"]))
        needed_pairs.add((str(row["participant_id"]), str(row["session_id"])))

print(f"Need features for {len(needed_pids)} participants, "
      f"{len(needed_pairs)} (participant, session) pairs\n")

# ── MFCC: already loaded as mfcc_df ──────────────────────────────
mfcc_df["participant_id"] = mfcc_df["participant_id"].astype(str)
mfcc_df["session_id"] = mfcc_df["session_id"].astype(str)

# Filter to only the pairs we need
mfcc_subset = mfcc_df[
    mfcc_df.apply(lambda r: (r["participant_id"], r["session_id"]) in needed_pairs, axis=1)
].copy()
print(f"MFCC subset: {mfcc_subset.shape}  "
      f"({mfcc_subset['participant_id'].nunique()} participants, "
      f"{mfcc_subset['session_id'].nunique()} sessions, "
      f"{mfcc_subset['task_name'].nunique() if 'task_name' in mfcc_subset.columns else '?'} tasks)")

# Read only the rows for our participants (filter after load since parquet
# doesn't support predicate pushdown on string columns easily)
mel_spec_df = pd.read_parquet(mel_spec_path)
mel_spec_df["participant_id"] = mel_spec_df["participant_id"].astype(str)
mel_spec_df["session_id"] = mel_spec_df["session_id"].astype(str)

mel_spec_subset = mel_spec_df[
    mel_spec_df.apply(lambda r: (r["participant_id"], r["session_id"]) in needed_pairs, axis=1)
].copy()
del mel_spec_df  # free memory

print(f"Mel spectrogram subset: {mel_spec_subset.shape}  "
      f"({mel_spec_subset['participant_id'].nunique()} participants, "
      f"{mel_spec_subset['session_id'].nunique()} sessions)")

# ── Merge MFCC + Mel Spectrogram into one features table ─────────
# Both share (participant_id, session_id, task_name) as key
features_df = mfcc_subset.merge(
    mel_spec_subset,
    on=["participant_id", "session_id", "task_name"],
    how="outer",
    suffixes=("_mfcc", "_mel"),
)
print(f"\nCombined features: {features_df.shape}")
print(f"  Columns: {list(features_df.columns)}")

# ── Join features with each disease table ─────────────────────────
disease_tables_with_features = {}

for disease_name, df in disease_tables.items():
    merged = df.merge(
        features_df,
        on=["participant_id", "session_id"],
        how="inner",
    )

    if len(merged) == 0:
        print(f"⚠  {disease_name:35s}  — no matching features found")
        continue

    disease_tables_with_features[disease_name] = merged

    n_pids = merged["participant_id"].nunique()
    n_sess = merged["session_id"].nunique()
    n_tasks = merged["task_name"].nunique() if "task_name" in merged.columns else 0
    has_mfcc = "mfcc" in merged.columns
    has_mel = "mel_spectrogram" in merged.columns

    print(f"✔  {disease_name:35s}  {merged.shape}  |  "
          f"{n_pids} pids, {n_sess} sessions, {n_tasks} tasks  "
          f"MFCC={'✔' if has_mfcc else '✗'}  MelSpec={'✔' if has_mel else '✗'}")

# ── Summary ───────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"  {len(disease_tables_with_features)} disease tables with audio features")
print(f"{'═' * 70}")
for disease_name, df in sorted(disease_tables_with_features.items()):
    n_pids = df["participant_id"].nunique()
    n_rows = len(df)

    # Count non-null features
    n_mfcc = df["mfcc"].notna().sum() if "mfcc" in df.columns else 0
    n_mel = df["mel_spectrogram"].notna().sum() if "mel_spectrogram" in df.columns else 0

    print(f"  {disease_name:35s}  {n_pids:>3d} participants, "
          f"{n_rows:>5d} rows  |  MFCC: {n_mfcc}  MelSpec: {n_mel}")

    # Show column breakdown
    diag_cols = [c for c in df.columns if c.startswith(f"{disease_name}__")]
    quest_cols = [c for c in df.columns
                  if "__" in c and not c.startswith(f"{disease_name}__")]
    feat_cols = ["task_name", "n_frames_mfcc", "mfcc",
                 "n_frames_mel", "mel_spectrogram"]
    feat_cols = [c for c in feat_cols if c in df.columns]
    print(f"    Diagnosis cols: {diag_cols}")
    print(f"    Quest cols ({len(quest_cols)}): {quest_cols[:3]}...")
    print(f"    Feature cols: {feat_cols}")
print(f"{'═' * 70}")

Need features for 8 participants, 25 (participant, session) pairs

MFCC subset: (812, 5)  (8 participants, 24 sessions, 105 tasks)

Loading mel spectrogram from torchaudio_mel_spectrogram.parquet ...
Mel spectrogram subset: (812, 5)  (8 participants, 24 sessions)

Combined features: (812, 7)
  Columns: ['participant_id', 'session_id', 'task_name', 'mfcc', 'n_frames_mfcc', 'mel_spectrogram', 'n_frames_mel']
✔  laryngeal_dystonia                   (44, 20)  |  1 pids, 1 sessions, 44 tasks  MFCC=✔  MelSpec=✔
✔  parkinsons_disease                   (700, 10)  |  6 pids, 20 sessions, 72 tasks  MFCC=✔  MelSpec=✔
✔  unilateral_vocal_fold_paralysis      (68, 21)  |  1 pids, 3 sessions, 34 tasks  MFCC=✔  MelSpec=✔

══════════════════════════════════════════════════════════════════════
  3 disease tables with audio features
══════════════════════════════════════════════════════════════════════
  laryngeal_dystonia                     1 participants,    44 rows  |  MFCC: 44  MelSpec: 44
    Diagn